In [1]:
from sql_agent import ask_sql

In [2]:
result = ask_sql("How many distinct users do we have?")

In [3]:
print(result.final_sql)
display(result.data)

SELECT COUNT(DISTINCT user_id) AS distinct_users
FROM dim_users;


,distinct_users
0,222


In [4]:
result = ask_sql("How many users are active on average per day?")

In [5]:
print(result.final_sql)
display(result.data)

SELECT AVG(daily_active_users) AS average_daily_active_users
FROM fact_daily_activity;


,average_daily_active_users
0,3.7257142857142857


In [6]:
result = ask_sql("How many users were active on average per day in july?")

In [7]:
print(result.final_sql)
display(result.data)

SELECT AVG(daily_active_users) AS average_daily_active_users
FROM fact_daily_activity
WHERE activity_date >= DATE '2023-07-01'
  AND activity_date < DATE '2023-08-01';


,average_daily_active_users
0,None


In [8]:
result = ask_sql("On what day did we have maximum number of users and what that the value?")

In [9]:
print(result.final_sql)
display(result.data)

SELECT activity_date, daily_active_users
FROM fact_daily_activity
ORDER BY daily_active_users DESC, activity_date ASC
LIMIT 1;


,activity_date,daily_active_users
0,2026-07-14,16


In [10]:
result = ask_sql("On what day did we have second maximum number of users and what that the value?")

In [11]:
print(result.final_sql)
display(result.data)

SELECT activity_date, daily_active_users
FROM fact_daily_activity
ORDER BY daily_active_users DESC, activity_date ASC
OFFSET 1
LIMIT 1;


,activity_date,daily_active_users
0,2026-08-16,13


In [12]:
result = ask_sql("How many users on average have at least two open_app events per day?")

In [13]:
print(result.final_sql)
display(result.data)

WITH activity_days AS (
    SELECT DISTINCT
        (e.client_ts AT TIME ZONE 'UTC')::DATE AS activity_date
    FROM events e
    JOIN dim_users u
        ON u.user_id = e.user_id
    WHERE u.is_test_user = FALSE
      AND e.client_ts IS NOT NULL
), qualifying_users_by_day AS (
    SELECT
        (e.client_ts AT TIME ZONE 'UTC')::DATE AS activity_date,
        e.user_id
    FROM events e
    JOIN dim_users u
        ON u.user_id = e.user_id
    WHERE e.action_type = 'open_app'
      AND u.is_test_user = FALSE
      AND e.client_ts IS NOT NULL
    GROUP BY
        (e.client_ts AT TIME ZONE 'UTC')::DATE,
        e.user_id
    HAVING COUNT(*) >= 2
), daily_qualifying_users AS (
    SELECT
        activity_date,
        COUNT(*) AS qualifying_user_count
    FROM qualifying_users_by_day
    GROUP BY activity_date
)
SELECT AVG(COALESCE(d.qualiFying_user_count, 0)) AS average_users_with_at_least_two_open_app_events_per_day
FROM activity_days a
LEFT JOIN daily_qualifying_users d
    ON d.activ

,average_users_with_at_least_two_open_app_events_per_day
0,0E-20


In [14]:
result = ask_sql("Show many users have at least two open_app events per day")

In [15]:
print(result.final_sql)
display(result.data)

SELECT
  q.activity_date,
  COUNT(*) AS users_with_at_least_two_open_app_events
FROM (
  SELECT
    e.user_id,
    (e.client_ts AT TIME ZONE 'UTC')::DATE AS activity_date
  FROM events e
  JOIN dim_users u
    ON u.user_id = e.user_id
  WHERE e.action_type = 'open_app'
    AND u.is_test_user = FALSE
  GROUP BY
    e.user_id,
    (e.client_ts AT TIME ZONE 'UTC')::DATE
  HAVING COUNT(*) >= 2
) AS q
GROUP BY q.activity_date
ORDER BY q.activity_date;


,activity_date,users_with_at_least_two_open_app_events


In [16]:
result = ask_sql("Show many users engage with missions per day")

if result.error:
    print(result.error)
else:
    display(result.data)
    print(result.final_sql)

,activity_date,mission_engaged_users


SELECT
  (e.client_ts AT TIME ZONE 'UTC')::DATE AS activity_date,
  COUNT(DISTINCT e.user_id) AS mission_engaged_users
FROM events e
JOIN dim_users u
  ON u.user_id = e.user_id
WHERE u.is_test_user = FALSE
  AND (
    e.action_type ILIKE '%mission%'
    OR e.action_subtype ILIKE '%mission%'
    OR e.action_json::text ILIKE '%mission%'
  )
GROUP BY (e.client_ts AT TIME ZONE 'UTC')::DATE
ORDER BY activity_date;


In [18]:
result = ask_sql("Show how many sweets is it OK to eat per day?")

if result.error:
    print(result.error)
else:
    display(result.data)
    print(result.final_sql)

The SQL reviewer could not approve a query for this question:
Show how many sweets is it OK to eat per day?

Reviewer issues:
- The proposed query returns daily in-app purchase metrics and diamonds purchased; it does not answer how many sweets are OK to eat per day.
- The database contains no dietary, nutritional, or sweets-consumption data, so the question cannot be answered from the available tables.
